# diseases.json 말미 잡데이터 제거

아산백과 상세 페이지의 `알기쉬운 의학용어`, 건강TV 동영상 시간/제목, footer 등이 section 말미에 섞였을 때 후처리하는 Notebook입니다.

In [1]:
import json, re
from pathlib import Path
from typing import Any, Dict, List, Tuple

INPUT_PATH = Path("output_amc_diseases_fixed/diseases.json")  # 필요하면 경로 수정
OUTPUT_PATH = Path("output_amc_diseases_fixed/diseases_cleaned.json")
REPORT_PATH = Path("output_amc_diseases_fixed/diseases_cleaning_report.json")

NOISE_EXACT = {
    "알기쉬운 의학용어", "건강TV", "콘텐츠 제공 문의하기", "다른질환보기",
    "질환백과", "질환설명", "의료진", "로그인", "회원가입", "나의차트",
    "병원둘러보기", "오시는길", "Language", "+ 더보기", "외",
}

NOISE_PREFIX = [
    "서울아산병원은 신뢰도 있는 건강정보 콘텐츠를 제공",
    "현재 페이지를 공유하기",
    "현재 페이지를 인쇄하기",
]

def norm_line(value: Any) -> str:
    value = str(value).replace("\xa0", " ").replace("\u200b", " ")
    return re.sub(r"[ \t]+", " ", value).strip()

def is_duration_split(lines: List[str], i: int) -> bool:
    line = norm_line(lines[i])
    if not re.fullmatch(r"\d{1,2}", line):
        return False
    if i + 2 >= len(lines):
        return False
    return norm_line(lines[i + 1]) == ":" and bool(re.fullmatch(r"\d{2}", norm_line(lines[i + 2])))

def should_stop(lines: List[str], i: int) -> bool:
    line = norm_line(lines[i])
    if not line:
        return False
    if line in NOISE_EXACT:
        return True
    if any(line.startswith(prefix) for prefix in NOISE_PREFIX):
        return True
    if re.fullmatch(r"\d{1,2}\s*:\s*\d{2}", line):
        return True
    if is_duration_split(lines, i):
        return True
    if "[건강플러스]" in line and len(line) < 90:
        return True
    return False

def clean_section_text(text: Any) -> Tuple[str, str]:
    if not isinstance(text, str) or not text.strip():
        return "", ""
    raw_lines = text.splitlines()
    kept, removed = [], []
    for i, line in enumerate(raw_lines):
        if should_stop(raw_lines, i):
            removed = raw_lines[i:]
            break
        kept.append(line)
    cleaned = "\n".join(norm_line(x) for x in kept).strip()
    removed_text = "\n".join(norm_line(x) for x in removed).strip()
    return cleaned, removed_text

def clean_diseases(diseases: List[Dict[str, Any]]):
    cleaned, report = [], []
    for disease in diseases:
        item = json.loads(json.dumps(disease, ensure_ascii=False))
        for section_key, section_text in list((item.get("sections") or {}).items()):
            cleaned_text, removed_text = clean_section_text(section_text)
            item["sections"][section_key] = cleaned_text
            if removed_text:
                report.append({
                    "content_id": item.get("content_id"),
                    "name_ko": item.get("name_ko"),
                    "section": section_key,
                    "removed_preview": removed_text[:300],
                })
        cleaned.append(item)
    return cleaned, report

# 실행
diseases = json.loads(INPUT_PATH.read_text(encoding="utf-8"))
cleaned, report = clean_diseases(diseases)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(cleaned, ensure_ascii=False, indent=2), encoding="utf-8")
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

print("원본 질환 수:", len(diseases))
print("정리 후 질환 수:", len(cleaned))
print("정리된 section 수:", len(report))
print("저장:", OUTPUT_PATH)
print("리포트:", REPORT_PATH)

# 남은 잡데이터 마커 검증
markers = ["알기쉬운 의학용어", "건강플러스", "콘텐츠 제공 문의하기", "다른질환보기", "로그인", "회원가입", "Language"]
remaining = []
for d in cleaned:
    for sec, text in (d.get("sections") or {}).items():
        for marker in markers:
            if marker in text:
                remaining.append((d.get("content_id"), d.get("name_ko"), sec, marker))
                break
print("남은 잡데이터 후보:", len(remaining))
if remaining[:10]:
    print(remaining[:10])


원본 질환 수: 63
정리 후 질환 수: 63
정리된 section 수: 40
저장: output_amc_diseases_fixed\diseases_cleaned.json
리포트: output_amc_diseases_fixed\diseases_cleaning_report.json
남은 잡데이터 후보: 0
